# Setup

In [5]:
from pprint import pprint
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from transformers import set_seed
from transformers.utils import logging

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Seed
seed = 42
set_seed(seed)

# Suppress loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# 1. Head-Only Fine-Tuning

## 1.1. Terminologies
- Weight decay
  - A regularization method that discourages weights from becoming unnecessarily large.
  - $w \leftarrow w - \eta \nabla L - \eta \lambda w$
    - $\eta$ = learning rate.
    - $\lambda$ = weight decay strength.
- Gradient clipping
  - Clipping L2-norm of the gradient under fixed value.
  - Prevents unusually large updates from destabilizing training.
- bf16
  - Less memory, faster training.
  - Usually stable enough for transformers fine-tuning.
  - `bf16=True` doesn't mean that every params become bf16. Some operations/state stay in higher precision if needed.

## 1.2. `Trainer`
| Method                                       | Description                                    | Returns                       |
| -------------------------------------------- | ---------------------------------------------- | ----------------------------- |
| `trainer.train()`                            | Start model training.                          | Training loss, metrics, steps |
| `trainer.train(resume_from_checkpoint=True)` | Resume training from the latest checkpoint.    | Training results              |
| `trainer.hyperparameter_search()`            | Perform HPO.                                   | Best parameters               |
| `trainer.evaluate()`                         | Evaluate on the validation dataset.            | Dictionary of metrics         |
| `trainer.predict(dataset)`                   | Generate predictions on a dataset.             | Predictions, labels, metrics  |
| `trainer.save_model(path)`                   | Save trained model and tokenizer.              | None                          |
| `trainer.save_state()`                       | Save Trainer state (e.g., steps, log history). | None                          |
| `trainer.save_metrics(split, metrics)`       | Save evaluation metrics to JSON.               | None                          |
| `trainer.push_to_hub()`                      | Upload model to Hugging Face Hub.              | Repository URL                |

| Attribute                             | Description                     |
| ------------------------------------- | ------------------------------- |
| `trainer.model`                       | Current model                   |
| `trainer.args`                        | TrainingArguments configuration |
| `trainer.state`                       | Current training state          |
| `trainer.state.log_history`           | Training and validation logs    |
| `trainer.state.best_metric`           | Best validation metric          |
| `trainer.state.best_model_checkpoint` | Path to best checkpoint         |
| `trainer.optimizer`                   | Optimizer used during training  |
| `trainer.lr_scheduler`                | Learning-rate scheduler         |


## 1.3. Example

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer, TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score

# Config
model_name = 'distilbert/distilbert-base-uncased'

# 1. Data
data = load_dataset('stanfordnlp/imdb')
split = data['train'].train_test_split(
    test_size=0.2,      # 5,000.
    shuffle=True,
    stratify_by_column='label',
    seed=seed,
)
ds = DatasetDict({
    'train': split['train'],
    'valid': split['test'],
    'test': data['test'],
})

# 2. Tokenization
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_len   = [len(tokenizer.encode(x)) for x in ds['train']['text']]
df_len      = pd.DataFrame(train_len)
percentiles = np.arange(0.0, 1.0, 0.1)
display(df_len.describe(percentiles=percentiles).tail())

max_length = 512
coverage = (np.array(train_len) <= max_length).mean()
print(f"{max_length} covers {coverage*100:.2f}% samples.")

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=max_length,
    )
ds_token = ds.map(tokenize, batched=True, batch_size=64, desc='Tokenizing train/valid/test')

# 3. Data collator
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)

for param in model.distilbert.parameters():
    param.requires_grad = False

# 5. Training
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,      # tolerate up to 2 consecutives without improvements > threshold.
    early_stopping_threshold=1e-3,  # should improve at least this value.
)

def compute_metrics(pred):
    """ (logits, labels) -> {'metric': metric} """
    logits, y_true = pred
    y_pred = logits.argmax(axis=-1)

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }

training_args = TrainingArguments(
    # Config
    output_dir='../tmp/outputs/imdb_frozen',
    num_train_epochs=3,                 # n_epochs.
    bf16=use_bf16,                      # use bf16 instead of fp16.
    per_device_train_batch_size=32,     # train_batch_size.
    per_device_eval_batch_size=32,      # valid_batch_size.
    gradient_accumulation_steps=1,      # accumulate gradients accross steps.
    disable_tqdm=False,                 # display tqdm.

    # Optimizer
    weight_decay=1e-2,              # slightly decrease weights per update.
    max_grad_norm=1.0,              # clipping L2-norm of the gradient.
    learning_rate=3e-4,             # relatively large for head-only tuning.
    warmup_steps=0.1,               # first 10% steps: 0 -> lr, linearly.
    lr_scheduler_type='linear',     # remaining steps: lr -> 0, depending on type.

    # Checkpoint
    eval_strategy='epoch',          # run validation after every epoch.
    save_strategy='epoch',          # save a checkpoint.
    logging_strategy='steps',       # report training loss.
    logging_steps=100,              # log training loss every 100 steps.
    save_total_limit=2,             # keep only the last 2 checkpoints.

    # Best model
    load_best_model_at_end=True,    # restore the best model, instead keeping the last one.
    metric_for_best_model='f1',     # evaluation metric.
    greater_is_better=True,         # direction.
)

trainer = Trainer(
    model=model,
    args=training_args,
    callbacks=[early_stopping],
    data_collator=collator,

    train_dataset=ds_token['train'],
    eval_dataset=ds_token['valid'],
    
    compute_metrics=compute_metrics,

    processing_class=tokenizer,     # to save together with model.
)

trainer.train(resume_from_checkpoint=False)

# 6. Learning curve
hist = pd.DataFrame(trainer.state.log_history)
train = hist.dropna(subset=['loss'])
valid = hist.dropna(subset=['eval_loss'])

plt.plot(train['step'], train['loss'], 'o-', label='Train')
plt.plot(valid['step'], valid['eval_loss'], 'o-', label='Valid')
plt.legend()
plt.grid()
plt.show()

# 7. Prediction
pred = trainer.predict(ds_token['test'])
y_pred = pred.predictions.argmax(axis=-1)
y_true = pred.label_ids

print(f"Test Accuracy: {accuracy_score(pred.label_ids, y_pred):.4f}")
print(f"Test F1: {f1_score(pred.label_ids, y_pred):.4f}")

# 8. Save & load
model_path = '../tmp/imdb_frozen'
trainer.save_model(model_path)

model_load = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer_load = AutoTokenizer.from_pretrained(model_path)

# 2. QLoRA

## 2.1. PEFT

### LoRA

- $W' = W + \Delta W$, where $W \in R^{d_{out} \times d_{in}}$
- $\Delta W = BA$, where $B \in R^{d_{out} \times r}, A \in R^{r \times d_{in}}, r \ll d_{out}, d_{in}$
- $y = Wx + \frac{\alpha}{r}BAx$, where $r = \text{rank}, \alpha = \text{lora alpha}$.

### Other methods

- Bottleneck adapters
  - Adds a small trainable network inside a transformer block.
  - vs LoRA: adds a low-rank update to an existing weight matrix.
- Prompt tuning
  - A transformer is frozen.
  - Hard prompt: adds readable tokens manually, e.g. "Classify the sentiment of this text: ".
  - Soft prompt: adds some additional embedding vectors before original embedding vectors.
- Prefix tuning
  - Inserts some learned continuous representations into the transformer's **attention** layers.
  - K = [prefix_K; normal_K]
  - V = [prefix_V; normal_V]
- $IA^3$
  - Instead of adding matrices $A, B$ like LoRA, it learns small scaling vectors -> even fewer parameters.
  - Hidden $h = [h_1, h_2, ..., h_d]$
  - Scaling vectors $l = [l_1, l_2, ..., l_d]$
  - Final hidden $h' = h \odot l$, $\odot$ = element-wise multiplication.
- AdaLoRA
  - Dynamically allocate $r$ according to the importance of each matrix.
- DoRA
  - Weight-Decomposed LoRA.
  - Decompose $W$ into magnitude and direction, i.e. $w = m \cdot u$ where $m = ||w||_2, u = \frac{w}{||w||_2}$
  - Magnitude -> learn directly with a small trainable parameter
  - Direction -> adapt using LoRA
  - `LoraConfig(use_dora=True)`

## 2.2. Quantization

### Basic

- Continuous values -> much smaller discrete representation.
  - e.g. [0.07, 0.26, 0.48] -> [0.0, 0.25, 0.5]
- $q = round(\frac{x}{s}) + z$
  - $q$ = stored quantized integer
  - $x$ = original value
  - $s$ = scale = $\frac{x_{max} - x_{min}}{q_{max} - q_{min}}$
    - e.g. range = [-1, 1], 4-bit unsigned ints = [0, 15] -> $s = \frac{1 - (-1)}{15 - 0} \approx 0.1333$
  - $z$ = zero-point = which quantized int corresponds to real value 0?
    - e.g. [-1, 1] -> [0, 15] -> $z \approx 8$
- $\hat{x} = s(q - z)$

### Symmetric vs Asymmetric

```markdown
[-1 ---------------- 0 ---------------- +1]
      half bins          half bins
```
- Symmetric quantization
  - Values are centered around 0.
  - $z = 0 \rightarrow q = round(\frac{x}{s})$
  - Real 0 maps directly to integer 0.
  - Very common for neural-network weights.

```markdown
[-0.2 ------------------------------ +1.0]
                all bins
```
- Asymmetric quantization
  - e.g. values in [-0.2, 1.0] -> [-1.0, -0.2] will be wasted in a symmetric quantization.
  - Can be useful for activations.

### Level of Quantization

- Weight quantization
  - $W_{fp16} \rightarrow W_{int4}$
- Activation quantization
  - Hidden states
  - Attention outputs
  - FFN activations
  - ...
- W8A8 = Weights into int8 / activations into int8
- W4A16 = Weights into int4 / activations into int16 = closer to QLoRA style.

### PTQ vs QAT

- PTQ: Post-Training Quantization
  - Train normally -> quantize weights.
  - Pros: simple/cheap -> very common for deployment/inference.
  - Cons: a model never learns quantization error -> probably lower performance.
- QAT: Quantization-Aware Training
  - During training, simulate the effects of quantization.
  - Let the model adapt to the quantization error during training.
  - Preserve accuracy better, but more complicated and expensive.
  - STE: straight-through estimator
    - Forward behaves like quantization.
    - Backward uses an approximate gradient.

### NF4

- NormalFloat 4-bit: introduced with QLoRA.
- Better representation than int4 for normally-distributed neural network's weights.
  - int4 → 16 uniformly distributed integer-like levels.
  - nf4 → 16 non-uniform levels for a normal distribution.

### Double Quant

- Quantize quantization constants, i.e. meta-data on quantization, again to save VRAM.

### QLoRA

- Quantization + LoRA
- $y = W_{\text{dequant(BF16)}}x_{\text{BF16}} + \frac{\alpha}{r}B_{\text{BF16}}A_{\text{BF16}}x_{\text{BF16}}$
- Store base weights as 4-bit -> temporarily dequantize to BF16/FP16 for computation.
- Base weights stay frozen, and only LoRA parameters are updated.

## 2.3. Example

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

# Quantization
skip_modules = ['pre_classifier', 'classifier']

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,      # W is dequantized into bf16.
    bnb_4bit_quant_type='nf4',                  # use 'nf4' instead of 'fp4'.
    bnb_4bit_use_double_quant=True,             # quantize quantization constants -> save VRAM.
    llm_int8_skip_modules=skip_modules,         # skip quantization (skip int4 modules too).
)
model_name = 'distilbert/distilbert-base-uncased'
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)
quantized_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,    # don't store gradients during forward; recompute during backprop -> reduce VRAM.
)

# LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,         # sequence classification.
    r=8,                                # higher r -> more trainable params -> greater capacity / higher VRAM.
    lora_alpha=16,                      # higher alpha -> stronger contribution from LoRA updates.
    lora_dropout=0.1,                   # drop 10% of the input to the LoRA adapter layer during training.
    target_modules='all-linear',        # target modules.
    exclude_modules=skip_modules,       # do not apply lora.
    modules_to_save=skip_modules,       # set trainable and save together with lora adapters.
    bias='none',                        # do not train base-layer biases.
)
model = get_peft_model(quantized_model, lora_config)

model.print_trainable_parameters()
memory_usage = model.get_memory_footprint() / 1024**3   # MB
print(f"Memory usage: {memory_usage:.4f} MB")

trainable params: 1,255,682 || all params: 68,210,692 || trainable%: 1.8409
Memory usage: 0.1157 MB


# 3. HPO

In [ ]:
# Model init
def model_init(trial=None):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        quantization_config=bnb_config,
    )
    quantized_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=True,
    )
    model = get_peft_model(quantized_model, lora_config)

    return model

trainer.model_init = model_init     # Optuna requires `model_init`.

# Objective
def compute_objective(metrics):
    return metrics['eval_f1']

# Search space
def hp_space(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-6, 5e-5, log=True)  # search by log scale.
    warmup_steps = trial.suggest_float('warmup_steps', 0.0, 0.2)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)

    return {
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'warmup_steps': warmup_steps,
    }

# Sampler: how to choose hyperparameter combinations
sampler = optuna.samplers.TPESampler(seed=seed)     # TPESampler: uses previous results to choose the next params more wisely.

# Pruner: early stopping for trials
pruner = optuna.pruners.MedianPruner(n_startup_trials=1)    # first 1 trials will be done without pruning.

best_run = trainer.hyperparameter_search(
    compute_objective=compute_objective,
    direction='maximize',
    backend='optuna',
    hp_space=hp_space,
    n_trials=2,
    sampler=sampler,
    pruner=pruner,
    n_jobs=1,               # run one trial at a time, since we use one GPU for now.
    gc_after_trial=True,    # garbage collection after each trial.
)
best_hp = best_run.hyperparameters
pprint(best_hp)

[I 2026-10-13 11:51:36,459] A new study created in memory with name: no-name-1afe5959-c538-4902-b430-8d73e4a88c1d
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.619709,0.542338,0.841600,0.840258
2,0.325798,0.327127,0.864400,0.862586
3,0.308196,0.320818,0.867200,0.866129


c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two v

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.315186,0.311202,0.872200,0.871713
2,0.275049,0.293788,0.882600,0.881294
3,0.267378,0.289115,0.885400,0.885331


c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two v

{'learning_rate': 1.0401663679887314e-05,
 'warmup_steps': 0.031203728088487304,
 'weight_decay': 0.00029375384576328325}


# 4. SFT

- Supervised Fine-Tuning
  - Pretraining: understanding a language.
  - Supervised fine-tuning: understanding a conversation.
- Roles
  - Inserted as a special token, model-specifically defined.
  - System: high-level behavior / policy / persona / task setup.
  - User: request or input.
  - Assistant: desired model response.

> Note) SFT also packs multiple samples, but each conversation should be preserved intact, not truncated.

## 4.1. Chat Template

- Chat template: structured message -> token sequence.
```python
messages = [
    {"role": "system", "content": "You are a concise math tutor."},
    {"role": "user", "content": "What is 2 + 3?"},
    {"role": "assistant", "content": "The answer is 5."},
]
```

$\downarrow$

```markdown
<|im_start|>system
You are a concise math tutor.<|im_end|>
<|im_start|>user
What is 2 + 3?<|im_end|>
<|im_start|>assistant
The answer is 5.<|im_end|>
```

In [3]:
from transformers import AutoTokenizer

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "You are a concise math tutor."},
    {"role": "user", "content": "What is 2 + 3?"},
    {"role": "assistant", "content": "The answer is 5."},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,                 # if True, return 'input_ids' and 'attention_mask'.
    add_generation_prompt=False,    # if True, append '<|im_start|>assistant\n' at the end. training=False, generation=True.
)

text

'<|im_start|>system\nYou are a concise math tutor.<|im_end|>\n<|im_start|>user\nWhat is 2 + 3?<|im_end|>\n<|im_start|>assistant\nThe answer is 5.<|im_end|>\n'

## 4.2. SFT Loss

- Full-sequence loss: learns to predict all sequence, i.e. <system>/<user>/<assistant>.
- Assistant-only (=Response-only) loss: only <assistant> can contribute to the loss.
- Completion-only: only the part specified for training, e.g. <user> -> <assistant> -> <user> -> <assistant> => only the last <assistant>.

In [4]:
from trl import SFTConfig

# Full-sequence loss
args = SFTConfig(
    assistant_only_loss=False,
    completion_only_loss=False,
)

# Assistant-only loss
args = SFTConfig(assistant_only_loss=True)

# Completion-only loss
args = SFTConfig(completion_only_loss=True)

## 4.3. Example

> Note) `SFTTrainer` automatically tokenizes -> no manual tokenization required.

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

# Data
data = load_dataset('HuggingFaceH4/ultrachat_200k')
ds = DatasetDict({
    'train': data['train_sft'].select(range(1_000)).select_columns(['messages']),    # keep it as Dataset for SFTTrainer compatibility.
    'test': data['test_sft'].select(range(1_000)).select_columns(['messages'])
})

# Tokenizer (no manual tokenization)
model_name = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Model
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)

# Training
sft_args = SFTConfig(
    output_dir='../tmp/outputs/qwen_sft',

    max_length=128,
    assistant_only_loss=True,   # ignores the labels for user tokens, i.e. -100.
    packing=False,              # if True, allow multiple examples into one sequence.
    packing_strategy='bfd',     # keep each example intact; do not split it across packed sequences.

    chat_template_path='Qwen/Qwen2.5-0.5B-Instruct',
    eos_token='<|im_end|>',

    learning_rate=1e-4,
    optim='paged_adamw_32bit',
    bf16=use_bf16,

    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,

    logging_strategy='steps',
    logging_steps=10,
    eval_strategy='epoch',
    disable_tqdm=False,
    report_to=[],
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    processing_class=tokenizer,
)

sft_trainer.train(resume_from_checkpoint=False)

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.694262,1.674484,1.662595,75520.000000,0.596354


TrainOutput(global_step=37, training_loss=1.8633807414286845, metrics={'train_runtime': 51.4593, 'train_samples_per_second': 11.465, 'train_steps_per_second': 0.719, 'total_flos': 162170792509440.0, 'train_loss': 1.8633807414286845, 'epoch': 1.0})